# Logic Programming in Python

We have been working through a number of examples of logic problems by hand. Such exercises can be fun and rewarding, and are certainly useful for developing our understanding. However, it is clearly not an approach that scales well and we would like to use the computer to solve logic problems.

The history of solving logic problems with the computer has a long and rich history. Noting that the earliest attempts to create AI systems were rooted in logic, and especially in **symbol manipulation**, we can look back on the history of computing and see major advances that were motivated by this problem. For example, the LISP programming language was specifically designed for symbol manipulation and introduced many innovations in programming languages including garbage collection, support for recursion, first-class functions, and proper conditional expressions not requiring a `goto`. It has had major influence on both AI and on programming language design. The *Prolog* language embodies the notions and concepts that we are studying here even more explicitly.

We are not using LISP or Prolog; we are using Python, and Python does not have any in-built symbolic manipulation or logical inference capabilities (it does include simple logical operators but these are of limited utility for our purposes. We must therefore use a library.

I have not found a library that I am particularly satisfied with. The best I have found is `kanren`, and that is what we shall use. It is not the most intuitive of packages, and the documentation has room for improvement, so we shall have to develop our understanding slowly.

Our first task is to install kanren. The version of kanren we shall use here is 0.2.3 and this can be found on [PyPI](https://pypi.org/project/kanren/). Kanren does not seem to be available in the Anaconda distribution, unfortunately but it is available via Pip. A possible alternative is `minikanren` which is available in Anaconda; however, I was not able to get this to work.

Let's first install the package. Before beginning, you should create a virtual environment for this module which will create a separate environment to keep it cleanly separated from other modules and thus avoid potential library version conflicts.

* Create a virtual environment in Anaconda. Please use Python version 3.13.5.
* Once you have created this environment, launch it, and install package `minikanren`.
* You will also need to install the `ipykernel` package for Jupyer to work properly.

We are now ready to begin solving some basic logic problem.

## Solving basic logic problems in Kanren

One of the most basic notions in kanren is the **logic variable**. These can be created very easily. The following cell imports the command for creating a logic variable and create a new logic variable called `x`.

In [63]:
from kanren import var
x = var()

This code doesn't do anything tangible, but it does set up the machinery that we need to solve problems. Here is one of the very simplest problems we can solve using logic programming. This does not yet look like a logic problem, but we should start on familiar territory:

* Find one number $x$ such that $x==5$.

In [64]:
from kanren import eq, run
run(1, x, eq(x,5))

(5,)

In Kanren, no computations are done until the `run` function is called. `run` takes three or more arguments:
* The number of results to return (0: all of them), in this example there is only one result.
* The logic variable for which we are trying to solve (`x` in this case)
* One or more **goals**. `eq` is one of kanren's built-in *goal constructor*.

Here is a more complex example that uses two logic variables and two goals. Can you understand what this code does and how it does it?

In [65]:
from kanren import vars
y, z = vars(2)
run(1, x, eq(x,z), eq(z,7))

(7,)

Another common goal constructor is `membero` which looks to see whether an item belongs to some collection. For example, the following example returns one (1) member `x` from the collection `mylist`:

In [66]:
from kanren import membero
mylist = [1,2,3,2,4,2,5]
run(1,x,membero(x,mylist))


(1,)

We can modify this easily to return two members of the collection:

In [67]:
run(2,x,membero(x,mylist))

(1, 2)

or three members of the collection:

In [68]:
run(3,x,membero(x,mylist))

(1, 2, 3)

or **all** members of the collection

In [69]:
run(0,x,membero(x,mylist))

(1, 2, 3, 2, 4, 2, 5)

Notice that `membero` returns the *set* of members of the collection and so does not contain duplicates.

## Some more goal constructors

`eq` and `membero` are two examples of kanren's inbuilt set of goal constructors. Here are two more:

* `lany`: is True if any of a specified set of sub-goals are True.
* `lall`: is True if all of the specified set of subgoals are True.
* `conde`: is True if all of some specified set of goals are True, or if all of some other specifified set of goals are True, or...

Let's see how these work. First, let's write a simple script to find numbers that occur in the first ten members of the prime number or the Fibonacci sequence. We wil first us `lany` to look for numbers that occur in either sequence, and then use `lall` to find numbers that appear in both sequences.

In [13]:
primes = [1,2,3,5,7,11,13,17,19,23]
fibonacci = [1,1,2,3,5,8,13,21,34,55]

# Find any number that is in either the Fibonacci sequence and the primes
from kanren import lany, run, membero, var
x = var()
solution = run(0,x, lany(membero(x,primes), membero(x,fibonacci)))
print(solution)

# Find any number that is in both the Fibonacci sequence and the primes
from kanren import lall
solution = run(0,x, lall(membero(x,primes), membero(x,fibonacci)))
print(solution)


(1, 1, 2, 1, 3, 2, 5, 3, 7, 5, 11, 8, 13, 13, 17, 21, 19, 34, 23, 55)
(1, 2, 3, 5, 13, 1)


Now let us use the `conde` goal constructor to find all numbers that are in either the first ten *odd* numbers and the first ten primes, or the first ten odds and the first ten Fibonacci numbers.

In [14]:
from kanren import conde
odds = [1,3,5,7,9,11,13,15,17,19]
solution = run(0, x,
               conde(
                   (membero(x,odds),membero(x,primes)),
                   (membero(x,odds),membero(x,fibonacci))
                   )
                )
print(solution)

(1, 1, 3, 3, 5, 5, 7, 13, 11, 1, 13, 17, 19)


These goal constructors are very powerful. Can you see an equivalence between them and the logic operators that we have become familar with?

Let us now see how we can use Kanren to solve natural deduction problems.

## Natural Deduction in Kanren

So far, we have used Kanren to reason about numerical types, but it is just as easy to reason about logical types. Recasting our trivial example  of *find $x$ such that x is equal to 5* in terms of logic, we can easily write:


In [ ]:
from kanren import run, var, eq
P = var()
run(0, P, eq(P, True))

(True,)

Let's use this to verify some simple proofs. Let's start with a really trivial one:

$P\land Q$

$\therefore P$

Here, we have one goal: $P\landQ$. How do we express this in kanren? Which of the goal constructors we have introduced could we use for this?

In [34]:
from kanren import run, var, eq, lall
P = var()
Q = var()
goals = lall(
    eq(P, True),
    eq(Q, True)
)
run(0, P, goals)

(True,)

Does this behave as you expect? Let's try another example:

$P\land Q$

$\therefore P$

In [35]:
from kanren import run, var, eq, lany
P = var()
Q = var()
goals = lany(
    eq(P, True),
    eq(Q, True)
)
run(0, P, goals)

(True, ~_2435)

What do you make of this? How should we interpret this output?

Kanren does not include goals/operators corresponding to material implication or negation so we will have to work out how to handle these.


In [ ]:
from kanren import eq, lany
P = var()
Q = var()

def implies(A,B):
    return lany(eq(A, False),eq(B, True))

rules = lall(
    implies(P,Q),
    eq(P, True),
)

run(0, Q, rules)

(True,)

## Relationships in Kanren

Kanren supports the definition of *relations* between things, and inference over those relations. 

In [70]:
from kanren import Relation, facts
bigger = Relation()
facts(bigger, ("Elephant", "Hippopotamus"),
      ("Hippopotamus", "Horse"),
      ("Elephant", "Horse"),
      ("Horse", "Dog"))

In terms of the logic operators we have been studying, a `Relation` in Kanren is essentially a material implication. A Kanren relation `bigger("Elephant", "Horse")` is roughly equivalent to 
`if Elephant(x) then x is bigger than Horse` and `if Horse(x) then Elephant is bigger than x`

In [71]:
run(1, x, bigger(x,"Hippopotamus"))

('Elephant',)

In [72]:
run(2, x, bigger(x,"Horse"))

('Elephant', 'Hippopotamus')

However, the rules aren't chained together so when we ask for animals bigger than a dog:

In [73]:
run(0, x, bigger(x,"Dog"))

('Horse',)

we only get "Horse". However, as with the earlier examples, we can build compound queries:

In [74]:
run(0, x, bigger(y,"Dog"), bigger(x,y))

('Elephant', 'Hippopotamus')

This would allow us to write a recursive query that found all animals bigger than a dog. We will not explore this, but will return to it when we study knowledge graphs in part 2 of the module.

## Other types of Goal Constructor
There are several other type of goal constructor available
* `lall` means that all specified goals have to be True (logical AND)
* `lany` means that any of the specified goals have to be True (logical OR)

So to get an animal that is bigger that a Hipppotamus or bigger than a Horse:

In [75]:
from kanren import lany, lall
run(0, x, lany(bigger(x,"Hippopotamus"), bigger(x,"Horse")))

('Elephant', 'Elephant', 'Hippopotamus')

An animal that is bigger than an Horse and bigger than a Hippopotamus:

In [76]:
run(0, x, lall(bigger(x,"Hippopotamus"), bigger(x,"Horse")))

('Elephant',)

Finally, if we wanted to find and animal that is bigger than a horse and smaller than elephant, or smaller than a horse we can use `conde`, which provides logical `AND-OR`.

In [77]:
from kanren import conde
run(0, x, conde(
    (bigger(x,"Horse"), bigger("Elephant",x)),
    (bigger("Horse",x),))
   )

('Hippopotamus', 'Dog')

The `help` for `conde` is quite useful here:

In [78]:
help(conde)

Help on function conde in module kanren.core:

conde(
    *goals: Union[Iterable[Callable[[Union[MutableMapping, Literal[False]]], Iterator[Union[MutableMapping, Literal[False]]]]], Iterator[Iterable[Callable[[Union[MutableMapping, Literal[False]]], Iterator[Union[MutableMapping, Literal[False]]]]]]]
) -> Union[Callable[[Union[MutableMapping, Literal[False]]], Iterator[Union[MutableMapping, Literal[False]]]], Iterator[Union[MutableMapping, Literal[False]]]]
    Form a disjunction of goal conjunctions.



## Worked Example

Let us now try to use Kanren to solve a Logic problem. We will use one that we have seen before.

Ahmed, Chen, and Niamh are three friends. They went for dinner together one evening. They decided that they wanted to try all of the items on the menu so they all chose a different main course and a different dessert. The menu options for the main course were Pizza, Daal, and Falafel. For dessert, the choices were Apple Pie, Cheesecake, and Ice Cream. Given the five statements below, can you work out who ordered what?

1. The person who had the Pizza did not have the Apple Pie.
1. Niamh had the Daal.
1. Ahmed did not have Cheesecake.
1. Chen had the Apple Pie.
1. The person who had the Ice Cream did not have the Falafel.

It isn't immediately obvious how to formulate this so let's begin by making some observations.
* We need all of these to be True so it seems likely that our goal will be constructed using `lall`.
* We have to be able to deal with negations, but **Kanren does not support direct negation** (there are good technical reasons for this). We may need to reformulate the problem to account for this.

Where do we even start? It isn't obvious, so let's define a skeleton that we can work with. We'll create a logical variable called `diners` that we will solve for, and a set of `rules`, all of which must be True

In [79]:
diners = var()

rules = lall()

solution = run(0, diners, rules)

print(solution)


(~_3521,)


This will turn out to be all that we need - except for the actual rules themselves. The rules need to define all of our knowledge of the solution. The logic solver will find sets of statements for which all of our rules hold.

First, we know that each solution needs to contain three logic variables (name, dinner, dessert). We need to tell the solver this.

In [80]:
from kanren import eq, run
rule = lall(eq(x,5))
run(1, x, rule)

(5,)

In [83]:
from kanren import eq
diners = var()

rules = lall(
    (eq((var(), var(), var()), diners))
)

solution = run(0, diners, rules)

print(solution)

((~_3531, ~_3532, ~_3533),)


Notice how this is formatted: the function and its arguments are passed separately. This is because we do not want the `eq` function to be evaluated at the point at which it is defined, Kanren's internals will combine them at the appropriate time.

Let's now add one of the easier clues: "Niamh had the Daal".

In [84]:
diners = var()

rules = lall(
    (eq((var(), var(), var()), diners)),
    (membero(("Niamh", "Daal", var()), diners))
)

solution = run(0, diners, rules)

print(solution)

((('Niamh', 'Daal', ~_3538), ~_3536, ~_3537), (~_3535, ('Niamh', 'Daal', ~_3538), ~_3537), (~_3535, ~_3536, ('Niamh', 'Daal', ~_3538)))


This line states that an entry in which Niamh had the Daal must be present in the solution. We see that this is indeed the case, but note that kanren has not yet been able to properly construct a solution because we have not yet specified the problem adequately.

Another easy clue is "Chen had the Apple Pie".

In [85]:
diners = var()

rules = lall(
    (eq((var(), var(), var()), diners)),
    (membero(("Niamh", "Daal", var()), diners)),
    (membero(("Chen", var(), "Apple Pie"), diners))
)

solution = run(0, diners, rules)

for s in solution:
    print(s)

(('Niamh', 'Daal', ~_3551), ('Chen', ~_3552, 'Apple Pie'), ~_3550)
(('Chen', ~_3552, 'Apple Pie'), ('Niamh', 'Daal', ~_3551), ~_3550)
(('Chen', ~_3552, 'Apple Pie'), ~_3549, ('Niamh', 'Daal', ~_3551))
(('Niamh', 'Daal', ~_3551), ~_3549, ('Chen', ~_3552, 'Apple Pie'))
(~_3548, ('Niamh', 'Daal', ~_3551), ('Chen', ~_3552, 'Apple Pie'))
(~_3548, ('Chen', ~_3552, 'Apple Pie'), ('Niamh', 'Daal', ~_3551))


Let's look now at one of the more complex clues which we cannot express quite so easily: "Ahmed did not have Cheesecake". Kanren does not have a `not` operator/goal constructor and so we need to formulate this differently. We can do this by noticing that this rule means that Ahmed must have had a dessert that was not Cheesecake, which means it must have been either Apple Pie or Ice Cream. We can express this using the `lany` goal constructor:

In [86]:
diners = var()

rules = lall(
    (eq((var(), var(), var()), diners)),
    (membero(("Niamh", "Daal", var()), diners)),
    (membero(("Chen", var(), "Apple Pie"), diners)),
    lany(
        (membero(("Ahmed",var(),"Apple Pie"),diners)),
        (membero(("Ahmed",var(),"Ice Cream"),diners))
    )
)

solution = run(0, diners, rules)

for s in solution:
    print(s)

(('Niamh', 'Daal', ~_3589), ('Chen', ~_3590, 'Apple Pie'), ('Ahmed', ~_3591, 'Apple Pie'))
(('Chen', ~_3590, 'Apple Pie'), ('Niamh', 'Daal', ~_3589), ('Ahmed', ~_3591, 'Apple Pie'))
(('Chen', ~_3590, 'Apple Pie'), ('Ahmed', ~_3591, 'Apple Pie'), ('Niamh', 'Daal', ~_3589))
(('Niamh', 'Daal', ~_3589), ('Ahmed', ~_3591, 'Apple Pie'), ('Chen', ~_3590, 'Apple Pie'))
(('Ahmed', ~_3591, 'Apple Pie'), ('Niamh', 'Daal', ~_3589), ('Chen', ~_3590, 'Apple Pie'))
(('Ahmed', ~_3591, 'Apple Pie'), ('Chen', ~_3590, 'Apple Pie'), ('Niamh', 'Daal', ~_3589))
(('Niamh', 'Daal', ~_3589), ('Chen', ~_3590, 'Apple Pie'), ('Ahmed', ~_3592, 'Ice Cream'))
(('Chen', ~_3590, 'Apple Pie'), ('Niamh', 'Daal', ~_3589), ('Ahmed', ~_3592, 'Ice Cream'))
(('Chen', ~_3590, 'Apple Pie'), ('Ahmed', ~_3592, 'Ice Cream'), ('Niamh', 'Daal', ~_3589))
(('Niamh', 'Daal', ~_3589), ('Ahmed', ~_3592, 'Ice Cream'), ('Chen', ~_3590, 'Apple Pie'))
(('Ahmed', ~_3592, 'Ice Cream'), ('Niamh', 'Daal', ~_3589), ('Chen', ~_3590, 'Apple Pie'))

We see now that we have a solution of the correct form but with some repetitions because we still have not completely specified the problem.

The other two clues can be expressed in the same way following reframing:
* "The person who had the Pizza did not have the Apple Pie" becomes "The person who had Pizza had Ice Cream or Cheesecake".
* "The person who had the Ice Cream did not have the Falafel" becomes " The person who had Ice Cream had Daal or Pizza".


In [87]:
diners = var()

rules = lall(
    (eq((var(), var(), var()), diners)),
    (membero(("Niamh", "Daal", var()), diners)),
    (membero(("Chen", var(), "Apple Pie"),diners)),
    lany(
        (membero((var(), "Pizza", "Cheesecake"), diners)),
        (membero((var(), "Pizza", "Ice Cream"), diners))
    ),
    lany(
        (membero(("Ahmed",var(),"Apple Pie"),diners)),
        (membero(("Ahmed",var(),"Ice Cream"),diners))
    ),
    lany(
        (membero((var(),"Falafel","Apple Pie"),diners)),
        (membero((var(),"Falafel","Cheesecake"),diners))
    )
)

solution = run(0, diners, rules)
for s in solution:
    print(s)

(('Niamh', 'Daal', ~_3725), ('Chen', 'Falafel', 'Apple Pie'), ('Ahmed', 'Pizza', 'Ice Cream'))
(('Chen', 'Falafel', 'Apple Pie'), ('Niamh', 'Daal', ~_3725), ('Ahmed', 'Pizza', 'Ice Cream'))
(('Chen', 'Falafel', 'Apple Pie'), ('Ahmed', 'Pizza', 'Ice Cream'), ('Niamh', 'Daal', ~_3725))
(('Niamh', 'Daal', ~_3725), ('Ahmed', 'Pizza', 'Ice Cream'), ('Chen', 'Falafel', 'Apple Pie'))
(('Ahmed', 'Pizza', 'Ice Cream'), ('Niamh', 'Daal', ~_3725), ('Chen', 'Falafel', 'Apple Pie'))
(('Ahmed', 'Pizza', 'Ice Cream'), ('Chen', 'Falafel', 'Apple Pie'), ('Niamh', 'Daal', ~_3725))


We now have an almost complete (albeit repetitive solution), but there is still a gap: what did Niamh have for dessert? Look back at how you solved this problem by hand. How did you deduce that Niamh had the Cheesecake for dessert? We used the constraint that everyone had to have ordered something, and having eliminated all of the other possibilities we deduced that this was the only remaining possibility.

We have not yet encoded that constraint into the rules. Let us do so now. The way we will do this is by adding constaints that every person, main course, and dessert must appear.

In [88]:
diners = var()

rules = lall(
    (eq((var(), var(), var()), diners)),
    lany(
        (membero((var(), "Pizza", "Cheesecake"), diners)),
        (membero((var(), "Pizza", "Ice Cream"), diners))
    ),
    (membero(("Niamh", "Daal", var()), diners)),
    lany(
        (membero(("Ahmed",var(),"Apple Pie"),diners)),
        (membero(("Ahmed",var(),"Ice Cream"),diners))
    ),
    (membero(("Chen", var(), "Apple Pie"),diners)),
    lany(
        (membero((var(),"Falafel","Apple Pie"),diners)),
        (membero((var(),"Falafel","Cheesecake"),diners))
    ),
    (membero(("Ahmed",var(),var()),diners)),
    (membero(("Niamh",var(),var()),diners)),
    (membero(("Chen",var(),var()),diners)),
    (membero((var(),"Falafel",var()),diners)),
    (membero((var(),"Pizza",var()),diners)),
    (membero((var(),"Daal",var()),diners)),
    (membero((var(),var(),"Cheesecake"),diners)),
    (membero((var(),var(),"Apple Pie"),diners)),
    (membero((var(),var(),"Ice Cream"),diners))
)

solution = run(0, diners, rules)
for s in solution:
    print(s)

(('Ahmed', 'Pizza', 'Ice Cream'), ('Niamh', 'Daal', 'Cheesecake'), ('Chen', 'Falafel', 'Apple Pie'))
(('Niamh', 'Daal', 'Cheesecake'), ('Ahmed', 'Pizza', 'Ice Cream'), ('Chen', 'Falafel', 'Apple Pie'))
(('Ahmed', 'Pizza', 'Ice Cream'), ('Chen', 'Falafel', 'Apple Pie'), ('Niamh', 'Daal', 'Cheesecake'))
(('Niamh', 'Daal', 'Cheesecake'), ('Chen', 'Falafel', 'Apple Pie'), ('Ahmed', 'Pizza', 'Ice Cream'))
(('Chen', 'Falafel', 'Apple Pie'), ('Ahmed', 'Pizza', 'Ice Cream'), ('Niamh', 'Daal', 'Cheesecake'))
(('Chen', 'Falafel', 'Apple Pie'), ('Niamh', 'Daal', 'Cheesecake'), ('Ahmed', 'Pizza', 'Ice Cream'))


This now uniquely solves the problem: the solutions are identical but just permuted.

Kanren can be used for solving any general logic problem, but one often has to put considerable thought into how to formulate the problem appropriately. Let's practice this.